# 02 — NLP & Semantic Recommendation Engine
## AI-Powered Natural-Language Product Recommendation System

This notebook demonstrates the core Natural Language Processing and Semantic Search pipeline:
1. **Query Understanding & Parsing**: Extracting product category, budget constraints, and specification keywords.
2. **Dense Vector Embeddings**: Using Pretrained Sentence Transformers (`all-MiniLM-L6-v2`) and vector representations.
3. **Cosine Similarity Computation**: Comparing customer query embeddings against Product Documents.
4. **Multi-Signal Hybrid Ranking**: Combining semantic similarity, budget fit, requirement fit, and discounts.
5. **Explainable AI**: Generating transparent natural-language reasons for every recommendation.


In [ ]:
import os
import sys
import pandas as pd
import numpy as np

sys.path.append(os.path.abspath('..'))
from src.data_preprocessing import load_product_catalog
from src.query_parser import parse_query, extract_budget, detect_category, extract_keywords
from src.semantic_recommender import SemanticRecommender
from src.ranking import HybridRecommender

catalog_df = load_product_catalog('../data/product_catalog.csv')
print(f"Catalog loaded with {len(catalog_df)} products.")

## 1. Natural Language Query Parsing & Entity Extraction

In [ ]:
test_queries = [
    "I need a laptop for Python and data analysis under ₹60,000",
    "Suggest a washing machine for a family under 40k",
    "I want a phone with a good camera under 30000",
    "1.5 Ton 5 Star split inverter AC for bedroom below 50k",
    "PS5 console for 4K gaming under ₹50,000",
    "Budget tablet for online study and notes under 20000"
]

print("=== Query Understanding Results ===\n")
for q in test_queries:
    parsed = parse_query(q)
    print(f"Query: '{q}'")
    print(f"  -> Category Detected : {parsed['category']}")
    print(f"  -> Budget Extracted  : ₹{parsed['budget']:,.0f}" if parsed['budget'] else "  -> Budget: None")
    print(f"  -> Key Specs/Features: {parsed['keywords']}")
    print("-" * 60)

## 2. Semantic Embedding & Recommender Engine Initialization

In [ ]:
semantic_engine = SemanticRecommender()
semantic_engine.fit_catalog(catalog_df)

hybrid_recommender = HybridRecommender(
    catalog_df=catalog_df,
    semantic_engine=semantic_engine
)
print("Hybrid recommender initialized successfully.")

## 3. End-to-End Recommendation & Ranking Demonstration

In [ ]:
sample_query = "I want a laptop with 16GB RAM for coding and data science under ₹60,000"
rec_result = hybrid_recommender.recommend(
    query=sample_query,
    top_k=3,
    strict_category=True
)

print(f"Query: {rec_result['query']}")
print(f"Total Candidate Pool: {rec_result['total_candidates']}")
print("\n--- Top 3 Recommendations ---")
for i, item in enumerate(rec_result['recommendations'], 1):
    print(f"\n#{i}: {item['item']} ({item['product_id']})")
    print(f"   Specs        : {item['specs']}")
    print(f"   Selling Price: ₹{item['selling_price']:,.0f} (MRP: ₹{item['mrp']:,.0f}, Save {item['discount_rate']:.1f}%)")
    print(f"   Score Matrix : Hybrid={item['hybrid_score']:.3f} | Semantic={item['semantic_score']:.3f} | BudgetFit={item['budget_fit']:.2f} | ReqFit={item['requirement_fit']:.2f}")
    print(f"   Explanation  : {item['explanation']}")

In [ ]:
# Testing across multiple customer intents
test_intents = [
    "Washing machine for a large family with steam wash under 45k",
    "Compact single door refrigerator for student room under 20000",
    "Portable bluetooth party speaker with deep bass"
]

for q in test_intents:
    res = hybrid_recommender.recommend(q, top_k=2)
    print(f"\n================ QUERY: '{q}' ================")
    for rank, p in enumerate(res['recommendations'], 1):
        print(f"  #{rank} [{p['category']}] {p['item']} | ₹{p['selling_price']:,.0f} | Match: {p['hybrid_score']*100:.1f}%")
        print(f"     Why: {p['explanation']}")